# Tangut strokes: what the data looks like, and how a score is made

Drawings made with a mouse on https://tangut.mkpo.li/dataset/, each with the Tangut character it was
made for. This notebook draws a few of them, runs the baseline submission through the scorer, and
shows where it loses.

In [ ]:
import json, urllib.request, glob
from pathlib import Path

# Prefer the mounted dataset; without a mount, read the same files from the public repository.
found = sorted(glob.glob("/kaggle/input/*/tangut-strokes.jsonl"))
if found:
    DATA = Path(found[0]).parent
    rows = [json.loads(line) for line in open(DATA / "tangut-strokes.jsonl", encoding="utf-8")]
    split = json.load(open(DATA / "split.json", encoding="utf-8"))
    baseline = [json.loads(line) for line in open(DATA / "baseline-submission.jsonl", encoding="utf-8")]
else:
    DATA = "https://huggingface.co/datasets/mkpoli/tangut-strokes/resolve/main"
    def fetch(name):
        request = urllib.request.Request(f"{DATA}/{name}",
                                         headers={"User-Agent": "kaggle-notebook/1.0 (tangut strokes example)"})
        with urllib.request.urlopen(request) as response:
            return response.read().decode()
    rows = [json.loads(line) for line in fetch("tangut-strokes.jsonl").splitlines() if line.strip()]
    split = json.loads(fetch("split.json"))
    baseline = [json.loads(line) for line in fetch("baseline-submission.jsonl").splitlines() if line.strip()]

answers = {row["id"]: row["char"] for row in rows}
print("source", DATA, "|", len(rows), "drawings,", len(split["test"]), "in the test half")
print(json.dumps({k: rows[0][k] for k in ("id", "char", "kind", "label", "width", "height")}, ensure_ascii=False))

## Drawings

Each record holds polylines in the unit square, in drawing order.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for ax, row in zip(axes.flat, rows[:12]):
    for stroke in row["strokes"]:
        xs = [x for x, _ in stroke]
        ys = [y for _, y in stroke]
        ax.plot(xs, ys, color="#a8402c", linewidth=1.6, solid_capstyle="round", solid_joinstyle="round")
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(1.05, -0.05)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    ax.set_title(row["char"], fontsize=16)
    for spine in ax.spines.values(): spine.set_visible(False)
fig.suptitle("Drawn on a pad with a mouse, beside the character it was made for", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# How scores are taken: rank of the answer in a short list, per test drawing.
test = set(split["test"])

def score(submission, answers, wanted):
    first = five = page = 0; reciprocal = 0.0; total = 0
    for entry in submission:
        if entry["id"] not in wanted or entry["id"] not in answers:
            continue
        ranked = entry["ranked"][:24]
        at = ranked.index(answers[entry["id"]]) if answers[entry["id"]] in ranked else -1
        total += 1
        first += at == 0
        five += 0 <= at < 5
        page += at >= 0
        reciprocal += 1 / (at + 1) if at >= 0 else 0
    return dict(n=total, top1=first/total, top5=five/total, top24=page/total, mrr=reciprocal/total)

print(score(baseline, answers, test))

## The baseline, and where it fails

The site's own matcher, across five typefaces. Two thirds of the answers are inside the first five
places; the misses below are the ones worth explaining.

In [ ]:
import random
misses = []
for entry in baseline:
    if entry["id"] not in test:
        continue
    answer = answers[entry["id"]]
    at = entry["ranked"].index(answer) if answer in entry["ranked"] else -1
    if at != 0:
        misses.append((entry["id"], answer, at, entry["ranked"][:3]))
print(len(misses), "test drawings where the answer was not first")

random.seed(1)
fig, axes = plt.subplots(1, 5, figsize=(13, 3.2))
by_id = {row["id"]: row for row in rows}
for ax, (ident, answer, at, top) in zip(axes, random.sample(misses, 5)):
    for stroke in by_id[ident]["strokes"]:
        ax.plot([x for x, _ in stroke], [y for _, y in stroke], color="#a8402c", linewidth=1.6)
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(1.05, -0.05); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"answer {answer} at {at + 1}\nfirst {''.join(top)}", fontsize=10)
    for spine in ax.spines.values(): spine.set_visible(False)
plt.tight_layout(); plt.show()

## Make an entry

Write `submissions/<name>.jsonl` with one line per test drawing and up to 24 characters, then run
`bun benchmark/score.mjs --name <name> submissions/<name>.jsonl` in the repository, or copy the
`score` function above. The split is fixed: 84 dev, 106 test.